In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile, os

os.makedirs('/content/hcmt', exist_ok=True)

with zipfile.ZipFile('/content/drive/MyDrive/tcga_brca_processed.zip', 'r') as z:
    z.extractall('/content/hcmt')

with zipfile.ZipFile('/content/drive/MyDrive/hcmt_src.zip', 'r') as z:
    z.extractall('/content/hcmt')

print("Files ready:", os.listdir('/content/hcmt'))

Files ready: ['requirements.txt', 'scripts', 'src', 'configs', 'data']


In [3]:
%pip install -q omegaconf wandb scikit-learn tqdm

In [4]:
path = '/content/hcmt/src/training/trainer.py'
with open(path) as f:
    content = f.read()

content = content.replace(
    'save_checkpoint(self.model, self.optimizer, epoch, val_metrics, str(best_ckpt_path))',
    'save_checkpoint(self.model, self.optimizer, self.scheduler, epoch, val_metrics, str(best_ckpt_path))'
)

with open(path, 'w') as f:
    f.write(content)

print("Fixed! Now rerun the training cell.")

Fixed! Now rerun the training cell.


In [5]:
# Fix 1: torch.load weights_only in utils.py
path = '/content/hcmt/src/utils/utils.py'
with open(path) as f:
    content = f.read()
content = content.replace(
    'ckpt = torch.load(path, map_location=device)',
    'ckpt = torch.load(path, map_location=device, weights_only=False)'
)
with open(path, 'w') as f:
    f.write(content)

# Fix 2: same fix in checkpoint.py
path2 = '/content/hcmt/src/utils/checkpoint.py'
with open(path2) as f:
    content2 = f.read()
content2 = content2.replace(
    'ckpt = torch.load(path, map_location=device)',
    'ckpt = torch.load(path, map_location=device, weights_only=False)'
)
with open(path2, 'w') as f:
    f.write(content2)

# Fix 3: disable AMP to prevent NaN loss
import re
cfg_path = '/content/hcmt/configs/default.yaml'
with open(cfg_path) as f:
    cfg = f.read()
cfg = cfg.replace('use_amp: true', 'use_amp: false')
with open(cfg_path, 'w') as f:
    f.write(cfg)

print("All fixed!")

All fixed!


In [8]:
# Fix the key name in utils.py
path = '/content/hcmt/src/utils/utils.py'
with open(path) as f:
    content = f.read()

content = content.replace(
    "model.load_state_dict(ckpt['model_state'])",
    "model.load_state_dict(ckpt['model_state_dict'])"
)

with open(path, 'w') as f:
    f.write(content)

print("Fixed!")

Fixed!


In [9]:
import os
os.chdir('/content/hcmt')
os.environ['WANDB_MODE'] = 'disabled'

!python3 scripts/train.py --config configs/default.yaml \
  data.use_wsi=false \
  data.use_radiology=false \
  data.label_column=PAM50 \
  experiment.name=genomics_clinical_gpu

2026-03-29 16:20:35.757145: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774801235.777630    4803 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774801235.784467    4803 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774801235.804847    4803 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774801235.804879    4803 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774801235.804884    4803 computation_placer.cc:177] computation placer alr

In [11]:
import os, glob

# Search for any .pt files
pt_files = glob.glob('/content/hcmt/**/*.pt', recursive=True)
print("All .pt files:", pt_files)

# Also check what's in checkpoints dir
print("Checkpoints dir:", os.listdir('/content/hcmt/checkpoints') if os.path.exists('/content/hcmt/checkpoints') else "doesn't exist")

# Check outputs dir too
print("Outputs dir:", os.listdir('/content/hcmt/outputs') if os.path.exists('/content/hcmt/outputs') else "doesn't exist")

All .pt files: ['/content/hcmt/data/tcga_brca/genomics/TCGA-E2-A15J.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-AR-A24U.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-E2-A3DX.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-AC-A3W5.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-AN-A049.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-B6-A0IJ.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-BH-A0H5.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-BH-A0HL.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-EW-A1J2.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-BH-A1FD.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-AC-A2FK.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-E9-A1R0.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-D8-A1XC.pt', '/content/hcmt/data/tcga_brca/genomics/f43b97fd-8953-4a5e-bafb-eb572145abe3.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-BH-A0C3.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-A1-A0SP.pt', '/content/hcmt/data/tcga_brca/genomics/TCGA-D8-A

In [ ]:
import shutil

shutil.copy(
    '/content/hcmt/checkpoints/genomics_clinical_gpu_best.pth',
    '/content/drive/MyDrive/hcmt_best_epoch26_f1_0.8127.pth'
)
print("Checkpoint saved to Google Drive!")